# Fund Liquidity Risk -- Numerical Examples

This notebook builds a complete liquidity risk framework for a Luxembourg AIF.
The fund is a mixed-asset open-ended AIF managed by a fully authorised AIFM.

**Regulatory basis:** Delegated Regulation 231/2013, Articles 46-49 (liquidity management);
AIFMD II Directive 2024/927/EU (LMTs); ESMA/2013/232 (Annex IV reporting).

**Contents:**
1. Portfolio construction -- mixed-asset AIF
2. Liquidity bucketing -- assign assets to AIFMD liquidation horizons
3. Investor base modelling -- redemption liability profile
4. Liquidity coverage -- matching assets to redemptions
5. Liquidity stress testing -- shocked scenarios
6. Annex IV liquidity reporting output
7. Liquidity Management Tools (LMTs) -- trigger analysis

In [ ]:
from quant_risk.setup import base

np, pd, plt = base()

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## 1. Portfolio Construction

The example fund is **Quant Alpha SICAV -- Multi-Asset AIF** (Luxembourg RAIF).

- Strategy: long-biased multi-asset -- large-cap equities, IG/HY corporate bonds, listed alternatives, private credit
- Redemption frequency: monthly, 30 business days notice
- Investor base: institutional (60%), family office (25%), retail professional (15%)

The portfolio is constructed at position level so liquidity bucketing is granular.

In [ ]:
positions = pd.read_csv('../../data/processed/aif_longbias_positions.csv')
positions['daily_volume'] = positions['daily_volume'].astype(float)
positions

In [ ]:
# ── fund parameters ──────────────────────────────────────────────────────────
NAV = positions['mv'].sum()   # EUR 460M, derived from actual positions

# here we can also calculate other fund-level parameters such as % equity, % redeemable, etc. if needed for the analysis
print(f"Total portfolio MV: EUR {positions['mv'].sum():,.0f}")
print(f"NAV: EUR {NAV:,.0f}")
print(f"MV as % of NAV: {positions['mv'].sum()/NAV*100:.1f}%")
positions[['name','asset_class','mv']].assign(
    pct_nav=lambda df: df['mv']/NAV*100
).style.format({'mv': '{:,.0f}', 'pct_nav': '{:.1f}%'})

## 2. Liquidity Bucketing

AIFMD Annex IV requires reporting the percentage of the portfolio that can be
liquidated within each horizon bucket. We assign positions using the
**10% participation rate convention**: a position takes
$\lceil MV / (0.10 \times DailyVolume) \rceil$ days to liquidate without
moving the market significantly.

**AIFMD Annex IV liquidity buckets:**

| Bucket | Horizon |
|--------|---------|
| 1 | 1 day |
| 2 | 2-7 days |
| 3 | 8-30 days |
| 4 | 31-90 days |
| 5 | 91-180 days |
| 6 | 181-365 days |
| 7 | > 1 year |

In [ ]:
PARTICIPATION_RATE = 0.10   # assume we trade at most 10% of daily market volume
BUCKET_LABELS = ['1 day', '2-7 days', '8-30 days', '31-90 days', '91-180 days', '181-365 days', '> 1 year']

def days_to_liquidate(mv, daily_volume, participation=PARTICIPATION_RATE):
    """How many trading days to fully liquidate a position."""
    if daily_volume == 0:
        return np.inf                            # illiquid -- no market
    daily_capacity = participation * daily_volume
    return np.ceil(mv / daily_capacity)

positions['days_to_liq'] = positions.apply(
    lambda r: days_to_liquidate(r['mv'], r['daily_volume']), axis=1
)
positions['bucket_label'] = pd.cut(
    positions['days_to_liq'],
    bins=[0, 1, 7, 30, 90, 180, 365, np.inf],
    labels=BUCKET_LABELS
)
positions['pct_nav']     = positions['mv'] / NAV * 100

print("Position-level liquidity bucketing:")
positions[['name','asset_class','mv','days_to_liq','bucket_label']]

In [ ]:
# ── aggregate by bucket -- reindex to all 7 so empty buckets show as zero ────
bucket_summary = (
    positions.groupby('bucket_label', observed=False)
    .agg(mv=('mv','sum'), pct_nav=('pct_nav','sum'))
    .reset_index()
)
bucket_summary['cumulative_pct'] = bucket_summary['pct_nav'].cumsum()

print("\nLiquidity profile -- AIFMD Annex IV buckets:")
print(bucket_summary[['bucket_label','mv','pct_nav','cumulative_pct']].to_string(index=False))


## 3. Investor Base Modelling -- Redemption Liability Profile

The redemption liability is not a single number -- it depends on who the
investors are and how they behave. We model three investor types:

| Type | AUM share | Historical redemption rate (normal) | Max redemption rate (stress) |
|------|-----------|-------------------------------------|-----------------------------|
| Institutional | 60% | 5% per month | 25% per month |
| Family office | 25% | 3% per month | 15% per month |
| Retail professional | 15% | 8% per month | 35% per month |

Redemption rate = % of that investor type's NAV redeemed per month.

The fund has a **30 business day notice period** and **monthly dealing**,
so the relevant horizon for redemption matching is 1 month (approximately
22 trading days).

In [ ]:
investors = pd.read_csv('../../data/processed/aif_longbias_investors.csv')
investors

In [ ]:
# calculate redemption amounts in EUR for each investor under normal and stress scenarios
investors['aum']                   = investors['share'] * NAV
investors['normal_redemption_eur'] = investors['aum'] * investors['normal_redemption']
investors['stress_redemption_eur'] = investors['aum'] * investors['stress_redemption']

redemptions_normal_total = investors['normal_redemption_eur'].sum()
redemptions_stress_total = investors['stress_redemption_eur'].sum()

print(f"\nNormal scenario total redemption: EUR {redemptions_normal_total:>15,.0f} ({redemptions_normal_total/NAV*100:.1f}% of NAV)")
      
print(f"Stress scenario total redemption: EUR {redemptions_stress_total:>15,.0f} ({redemptions_stress_total/NAV*100:.1f}% of NAV)")

print("\nInvestor base redemption profile (monthly):")
investors[['type','normal_redemption_eur','stress_redemption_eur']]


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# bar chart -- % NAV per bucket
axes[0].bar(bucket_summary['bucket_label'], bucket_summary['pct_nav'],
             edgecolor='white')
axes[0].set_title('Portfolio Liquidity Profile (% NAV per bucket)', fontsize=11)
axes[0].set_ylabel('% NAV')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(bucket_summary['pct_nav']):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=9)

# cumulative liquidation curve
axes[1].plot(bucket_summary['bucket_label'], bucket_summary['cumulative_pct'],
             marker='o',  linewidth=2)
axes[1].axhline(100, color='grey', linestyle='--', linewidth=1)
axes[1].set_title('Cumulative Liquidation Curve (% NAV)', fontsize=11)
axes[1].set_ylabel('Cumulative % NAV')
axes[1].tick_params(axis='x', rotation=30)
axes[1].set_ylim(0, 110)
for i, v in enumerate(bucket_summary['cumulative_pct']):
    axes[1].text(i, v + 1.5, f'{v:.1f}%', ha='center', fontsize=9)
plt.tight_layout()

## 4. Liquidity Coverage -- Matching Assets to Redemptions

The core question: can the fund liquidate enough assets within the notice period
to meet redemptions without selling illiquid positions at distressed prices?

We match cumulative liquidatable assets (by bucket) against the redemption
liability under normal and stress scenarios.

**Redemption horizon:** 30 business days = bucket 3 (8-30 days) or better.
This means all assets in buckets 1, 2, and 3 are available to meet redemptions
within the notice period.

In [ ]:
# ── assets available within the notice period (buckets 1-3, i.e. <= 30 days) ─
liquid_buckets = ['1 day', '2-7 days', '8-30 days']
available_30d = positions[positions['bucket_label'].isin(liquid_buckets)]['mv'].sum()
available_30d_pct = available_30d / NAV * 100

# ── liquidity coverage ratio ──────────────────────────────────────────────────
# LCR(fund) = liquid assets within horizon / redemption liability
lcr_normal = available_30d / redemptions_normal_total
lcr_stress = available_30d / redemptions_stress_total

print("Liquidity Coverage Analysis (30-day horizon):")
print(f"  Assets liquidatable within 30 days:   EUR {available_30d:>15,.0f}  ({available_30d_pct:.1f}% of NAV)")
print(f"  Normal redemption liability (monthly): EUR {redemptions_normal_total:>15,.0f}  ({redemptions_normal_total/NAV*100:.1f}% of NAV)")
print(f"  Stress redemption liability (monthly): EUR {redemptions_stress_total:>15,.0f}  ({redemptions_stress_total/NAV*100:.1f}% of NAV)")
print()
print(f"  Fund LCR -- normal scenario: {lcr_normal:.2f}x  {'PASS' if lcr_normal >= 1 else 'FAIL'}")
print(f"  Fund LCR -- stress scenario: {lcr_stress:.2f}x  {'PASS' if lcr_stress >= 1 else 'FAIL'}")

# ── coverage at each bucket ───────────────────────────────────────────────────
coverage = bucket_summary.copy()
coverage['normal_surplus'] = coverage['cumulative_pct']/100 * NAV - redemptions_normal_total
coverage['stress_surplus'] = coverage['cumulative_pct']/100 * NAV - redemptions_stress_total

print("\nCoverage by liquidation horizon:")
print(coverage[['bucket_label','cumulative_pct','normal_surplus','stress_surplus']].to_string(index=False))

In [ ]:
# ── coverage waterfall chart ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

x = range(len(bucket_summary))

ax.bar(x, bucket_summary['cumulative_pct'], label='Cumulative liquidatable (% NAV)')

ax.axhline(redemptions_normal_total/NAV*100, color='blue', linestyle='--', linewidth=2,
           label=f'Normal redemption ({redemptions_normal_total/NAV*100:.1f}% NAV)')
ax.axhline(redemptions_stress_total/NAV*100, color='red', linestyle='--', linewidth=2,
           label=f'Stress redemption ({redemptions_stress_total/NAV*100:.1f}% NAV)')

ax.set_xticks(list(x))
ax.set_xticklabels(bucket_summary['bucket_label'])
ax.set_ylabel('Cumulative % NAV')
ax.set_title('Liquidity Coverage -- Cumulative Assets vs Redemption Liability', fontsize=11)
ax.legend()
ax.set_ylim(0, 115)

plt.tight_layout()


## 5. Liquidity Stress Testing

Per Article 48 of Delegated Regulation 231/2013, AIFMs must conduct liquidity
stress tests covering both asset-side and liability-side shocks jointly.

We run three scenarios:

| Scenario | Asset shock | Redemption shock |
|----------|-------------|------------------|
| Base | No shock | Normal redemptions |
| Market stress | Daily volumes -50%, HY bonds -70% | Normal redemptions |
| Redemption stress | No shock | 2x normal redemptions |
| Combined stress | Daily volumes -50%, HY bonds -70% | Stress redemptions |

Asset shock mechanics:
- Equities and govt bonds: daily volume reduced by 50% (wider bid-ask, thinner book)
- HY bonds: daily volume reduced by 70% (credit stress -- dealers pull back)
- Private credit: remains illiquid (no change)
- Cash: unaffected

In [ ]:
def apply_asset_shock(positions_df, haircut_equity=0.0, haircut_hy=0.0):
    """Apply volume haircuts to simulate market stress."""
    df = positions_df.copy()

    equity_mask = df['asset_class'].str.contains('Equity|Govt Bond|Corp Bond -- IG|Cash|Listed')
    hy_mask     = df['asset_class'].str.contains('Corp Bond -- HY')

    df.loc[equity_mask, 'daily_volume'] *= (1 - haircut_equity)
    df.loc[hy_mask,     'daily_volume'] *= (1 - haircut_hy)
    
    result = df.groupby('bucket_label').agg(mv=('mv','sum'), pct_nav=('pct_nav','sum')).sort_values('bucket_label').reset_index()

    return result

scenarios = {
    'Base':              {'haircut_eq': 0.00, 'haircut_hy': 0.00, 'redemption': redemptions_normal_total},
    'Market stress':     {'haircut_eq': 0.50, 'haircut_hy': 0.70, 'redemption': redemptions_normal_total},
    'Redemption stress': {'haircut_eq': 0.00, 'haircut_hy': 0.00, 'redemption': redemptions_stress_total},
    'Combined stress':   {'haircut_eq': 0.50, 'haircut_hy': 0.70, 'redemption': redemptions_stress_total},
}

results = {}
for scenario_name, params in scenarios.items():
    shocked = apply_asset_shock(positions, params['haircut_eq'], params['haircut_hy'])
    avail_30d = shocked[shocked['bucket_label'].isin(liquid_buckets)]['mv'].sum()
    lcr = avail_30d / params['redemption']
    results[scenario_name] = {
        'available_30d_eur': avail_30d,
        'available_30d_pct': avail_30d / NAV * 100,
        'redemption_eur':    params['redemption'],
        'redemption_pct':    params['redemption'] / NAV * 100,
        'lcr':               lcr,
        'pass':              lcr >= 1.0,
    }

results_df = pd.DataFrame(results).T
print("Liquidity Stress Test Results:")
print(results_df.to_string())

In [ ]:
# ── stress test visualisation ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))


scenario_names = list(results.keys())
lcr_values     = [results[s]['lcr'] for s in scenario_names]
colors         = ['blue' if r['pass'] else 'red' for r in results.values()]

bars = ax.bar(scenario_names, lcr_values, color=colors, alpha=.7, width=0.5)
ax.axhline(1.0, color='black', linestyle='--', linewidth=1.5, label='Minimum coverage = 1.0x')

for bar, val in zip(bars, lcr_values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.03,
            f'{val:.2f}x', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel('Liquidity Coverage Ratio (LCR)')
ax.set_title('Fund LCR by Stress Scenario -- 30-day horizon', fontsize=11)
ax.legend()
ax.set_ylim(0, max(lcr_values) * 1.2)

# colour legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='blue', alpha=.7, label='Pass (LCR >= 1.0x)'),
    Patch(color='red', alpha=.7, label='Fail (LCR < 1.0x)'),
    plt.Line2D([0],[0], color='black', linestyle='--', label='Minimum = 1.0x')
])

plt.tight_layout()


## 6. Annex IV Liquidity Reporting Output

AIFMD Annex IV requires AIFMs to report the liquidity profile of each AIF
to the national competent authority (CSSF for Luxembourg funds).

The liquidity section of Annex IV reports the **percentage of the AIF's
portfolio** falling into each prescribed liquidation bucket, plus the
**percentage of investor equity** that could demand redemption in each
period (the liability side).

We generate the output in the format expected by the CSSF submission.

In [ ]:
annex_iv_asset = bucket_summary[['bucket_label','pct_nav']].copy()
annex_iv_asset.columns = ['Liquidation horizon', 'pct of portfolio (base)']

shocked_combined = apply_asset_shock(positions, 0.50, 0.70)

annex_iv_asset = annex_iv_asset.merge(
    shocked_combined[['bucket_label', 'pct_nav']].rename(columns={'pct_nav': 'pct of portfolio (stressed)'}),
    left_on='Liquidation horizon',
    right_on='bucket_label',
    how='left'
).drop(columns='bucket_label').fillna(0)

print("AIFMD Annex IV -- Portfolio Liquidity Profile:")
print(annex_iv_asset.to_string(index=False))
print(f"\nTotal: {annex_iv_asset['pct of portfolio (base)'].sum():.1f}% (base) | "
      f"{annex_iv_asset['pct of portfolio (stressed)'].sum():.1f}% (stressed)")

annex_iv_liability['pct of AIF equity'].values

In [ ]:
# ── Annex IV investor redemption profile (liability side) ─────────────────────
# Reports: what % of AIF equity could be redeemed within each notice period bucket
# Based on contractual redemption terms per investor type

# Our fund: monthly dealing, 30 business days notice
# All investor equity is therefore in the 8-30 day bucket (notice period)

def make_liability_profile(bucket_labels, allocations: dict) -> pd.DataFrame:
    """
    Build Annex IV liability profile from a sparse allocation dict.
    allocations: {bucket_label: pct_of_aif_equity} -- missing buckets get 0.
    Raises ValueError if allocations do not sum to 100.
    """
    total = sum(allocations.values())
    if not np.isclose(total, 100.0):
        raise ValueError(f"Allocations sum to {total:.1f}%, must be 100%")
    return pd.DataFrame({
        'Notice period':      bucket_labels,
        'pct of AIF equity':  [allocations.get(b, 0.0) for b in bucket_labels],
    })

# monthly dealing, 30 business days notice -- all equity redeemable in 8-30 day bucket
annex_iv_liability = make_liability_profile(
    BUCKET_LABELS,
    {'8-30 days': 100.0}
)

print("AIFMD Annex IV -- Investor Redemption Profile (contractual):")
print(annex_iv_liability.to_string(index=False))

# ── full Annex IV liquidity section side by side ──────────────────────────────
annex_iv = annex_iv_asset.copy()
annex_iv['pct equity redeemable'] = annex_iv_liability['pct of AIF equity'].values
annex_iv['liquidity gap (base)'] = annex_iv['pct of portfolio (base)'] - annex_iv['pct equity redeemable']

print("\nAIFMD Annex IV -- Full Liquidity Section (base scenario):")
print(annex_iv.to_string(index=False))
print("\nPositive gap = more liquidatable assets than redeemable equity in that bucket.")
print("Negative gap = potential mismatch -- must be covered by earlier buckets.")

In [ ]:
# ── Annex IV visualisation ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_prop_cycle(color=plt.cm.Blues(np.linspace(0.4, 0.85, 3)))

x = np.arange(len(annex_iv))
width = 0.28

ax.bar(x - width, annex_iv['pct of portfolio (base)'],    width, label='Portfolio (base)')
ax.bar(x,         annex_iv['pct of portfolio (stressed)'], width, label='Portfolio (stressed)')
ax.bar(x + width, annex_iv['pct equity redeemable'],       width, label='Investor equity redeemable')

ax.set_xticks(x)
ax.set_xticklabels(annex_iv['Liquidation horizon'])
ax.set_ylabel('pct of NAV / AIF equity')
ax.set_title('AIFMD Annex IV -- Liquidity Profile: Assets vs Redemption Rights', fontsize=11)
ax.legend()

plt.tight_layout()


## 7. Liquidity Management Tools (LMTs) -- Trigger Analysis

Under AIFMD II (Directive 2024/927/EU, Annex V), open-ended AIFs must have
at least two LMTs available. The AIFM must define conditions for activation
in its LMT policy (Del. Reg. 231/2013, Art. 95).

We model three LMTs relevant to this fund:

| LMT | Type | Trigger | Action |
|-----|------|---------|--------|
| Redemption gate | Quantitative -- automatic | Redemptions > 10% of NAV in one dealing period | Cap redemptions at 10% NAV; excess deferred to next period |
| Swing pricing | Anti-dilution tool | Net flows > 5% of NAV | Adjust NAV by 50bps swing factor to protect remaining investors |
| Suspension | Exceptional -- board decision | Gate active 2+ consecutive months AND backlog > 50% of liquid sleeve | Full suspension of subscriptions and redemptions |

Two additional dynamics modelled:

- **Deferred queue carry-forward:** redemptions not paid in month T due to the
  gate re-enter the queue in month T+1 on top of new requests
- **Contagion multiplier:** when a gate fires, new redemption requests in the
  following month are amplified by 1.5x -- investors who observed the gate
  submit fresh orders anticipating further stress

We simulate 12 months including two injected stress months (M5: 14% NAV, M10: 22% NAV).

### Suspension 

Unlike the gate and swing pricing, suspension has no fixed quantitative trigger
under AIFMD II. Per ESMA Guidelines (ESMA34-671404336-1364, April 2025) and
Directive 2024/927/EU Annex V, suspension is reserved for exceptional
circumstances and requires a board decision -- it cannot be activated
automatically.

In practice, suspension combines multiple conditions:

- Gate has been active for 2 or more consecutive months -- the deferred queue
  is not clearing
- Backlog exceeds a material fraction of the remaining liquid sleeve -- the fund
  cannot service the queue even at full gate capacity
- Market conditions prevent asset liquidation at reasonable prices

The AIFM must document suspension conditions in its LMT policy (Del. Reg.
231/2013, Art. 95) and notify the CSSF immediately upon activation.

**Implementation here:** `suspension_active` fires when the gate has been active
for at least 2 consecutive months AND the backlog exceeds 50% of the remaining
liquid sleeve. This reflects the board-decision nature of suspension -- a single
stress month is managed by the gate alone, but a sustained liquidity crisis
triggers the more severe tool.

In [ ]:
np.random.seed(42)

# ── simulate 12 months of monthly redemption flows ────────────────────────────
n_months = 12

# base redemption rate draws -- lognormal centred around normal_total/NAV
base_rate   = redemptions_normal_total / NAV          
normal_draw = np.random.lognormal(
    mean=np.log(base_rate) - 0.3,
    sigma=0.5,
    size=n_months
)
# inject two stress months
normal_draw[4] = 0.14    # month 5: large redemption wave
normal_draw[9] = 0.22    # month 10: extreme outflow

months = [f'M{i+1}' for i in range(n_months)]

# ── LMT trigger logic ─────────────────────────────────────────────────────────
GATE_THRESHOLD  = 0.10   # gate activates if redemptions > 10% NAV
SWING_THRESHOLD = 0.05   # swing pricing activates if net flows > 5% NAV

# liquid assets as a fraction of NAV -- from section 4
liquid_fraction = available_30d / NAV   # buckets 1-3, computed earlier

nav     = float(NAV)
backlog = 0.0

rows = []
for month, gross in zip(months, normal_draw):
    liquid_available = liquid_fraction * nav        # liquid assets scale with NAV
    requested        = gross * nav                  # EUR requested this month
    gate_limit       = GATE_THRESHOLD * nav         # max payable this month
    paid             = min(requested, gate_limit)   # actual outflow
    backlog         += requested - paid             # deferred queue grows
    nav             -= paid                         # NAV shrinks by what was paid

    gate_active      = requested > gate_limit
    swing_active     = gross > SWING_THRESHOLD
    swing_factor_bps = 50.0 if swing_active else 0.0
    # suspension: backlog exceeds what liquid assets can cover
    suspension_active = backlog > liquid_available

    rows.append({
        'month':                   month,
        'nav_eur':                 nav,
        'gross_redemption_pct':    gross * 100,
        'requested_eur':           requested,
        'paid_eur':                paid,
        'backlog_eur':             backlog,
        'backlog_pct_nav':         backlog / nav * 100,
        'liquid_available_eur':    liquid_available,
        'gate_active':             gate_active,
        'swing_active':            swing_active,
        'swing_factor_bps':        swing_factor_bps,
        'suspension_active':       suspension_active,
    })

lmt_df = pd.DataFrame(rows)

print("LMT Trigger Simulation -- 12-month flow:")
def highlight_lmt(val):
    if val is True:  return 'color: #b71c1c; font-weight: bold'  # red
    if val is False: return 'color: #1b5e20; font-weight: bold'  # green
    return ''

lmt_df.style.map(highlight_lmt, subset=['gate_active', 'swing_active', 'suspension_active'])

In [ ]:
np.random.seed(42)

# ── simulate 12 months of monthly redemption flows ────────────────────────────
n_months = 12

base_rate   = redemptions_normal_total / NAV
normal_draw = np.random.lognormal(
    mean=np.log(base_rate) - 0.3,
    sigma=0.5,
    size=n_months
)
normal_draw[4] = 0.14
normal_draw[9] = 0.22

months = [f'M{i+1}' for i in range(n_months)]

# ── LMT parameters ────────────────────────────────────────────────────────────
GATE_THRESHOLD       = 0.10
SWING_THRESHOLD      = 0.05
CONTAGION_MULTIPLIER = 1.50
SUSPENSION_BACKLOG   = 0.50
SUSPENSION_CONSEC    = 2

# ── sleeve tracking -- illiquid never sold, liquid depletes first ──────────────
liquid_nav   = float(available_30d)
illiquid_nav = float(NAV - available_30d)   # private credit -- fixed, cannot be sold
nav          = liquid_nav + illiquid_nav
backlog      = 0.0
gate_consec  = 0

rows = []
for i, (month, gross) in enumerate(zip(months, normal_draw)):
    if i > 0 and rows[i-1]['gate_active']:
        gross = gross * CONTAGION_MULTIPLIER

    nav_before   = nav

    requested    = gross * nav_before + backlog
    gate_limit   = GATE_THRESHOLD * nav_before
    paid         = min(requested, gate_limit)
    backlog      = requested - paid

    # only liquid sleeve depletes -- illiquid cannot be sold quickly
    liquid_nav   = max(0.0, liquid_nav - paid)
    nav          = liquid_nav + illiquid_nav   # reflects true composition

    gate_active  = requested > gate_limit
    gate_consec  = gate_consec + 1 if gate_active else 0

    suspension_active = (
        gate_consec >= SUSPENSION_CONSEC
        and backlog > SUSPENSION_BACKLOG * liquid_nav
    )

    rows.append({
        'month':                month,
        'nav_eur':              nav,
        'liquid_nav_eur':       liquid_nav,
        'illiquid_nav_eur':     illiquid_nav,
        'liquid_pct_nav':       liquid_nav / nav * 100 if nav > 0 else 0,
        'gross_redemption_pct': gross * 100,
        'requested_eur':        requested,
        'requested_pct':        requested / nav_before * 100,
        'gate_limit_pct':       gate_limit / nav_before * 100,
        'paid_eur':             paid,
        'backlog_eur':          backlog,
        'backlog_pct_nav':      backlog / nav * 100 if nav > 0 else 0,
        'gate_active':          gate_active,
        'gate_consecutive':     gate_consec,
        'swing_active':         gross > SWING_THRESHOLD,
        'swing_factor_bps':     50.0 if gross > SWING_THRESHOLD else 0.0,
        'suspension_active':    suspension_active,
    })

lmt_df = pd.DataFrame(rows)

# ── plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)

# top panel -- redemption bars
ax = axes[0]
gate_pct     = lmt_df['gate_limit_pct']
deferred_pct = (lmt_df['requested_pct'] - gate_pct).clip(lower=0)
paid_pct     = lmt_df['requested_pct'] - deferred_pct

ax.bar(months, paid_pct,
       color='blue', edgecolor='white', label='Paid')
ax.bar(months, deferred_pct,
       bottom=paid_pct, color='red', edgecolor='white', label='Deferred (queued, gate applied)')
ax.plot(months, lmt_df['gate_limit_pct'],
        color='black', linestyle='--', linewidth=1.5,
        label='Gate threshold (10% of current NAV)')

for i, (swing, bps) in enumerate(zip(lmt_df['swing_active'], lmt_df['swing_factor_bps'])):
    if swing:
        ax.text(i, lmt_df['requested_pct'].iloc[i] + 0.3,
                f'Swing\n{bps:.0f}bps', ha='center', color='darkblue')

ax.set_ylabel('Redemptions (% NAV)')
ax.set_title('LMT Trigger Analysis -- 12-month Redemption Simulation')
ax.legend()
ax.spines['bottom'].set_color('#555555')
ax.grid(axis='y')

# bottom panel -- NAV decomposition and backlog
ax2 = axes[1]

ax2.stackplot(months,
              lmt_df['liquid_nav_eur'] / 1e6,
              lmt_df['illiquid_nav_eur'] / 1e6,
              labels=['Liquid sleeve (EUR m)', 'Illiquid sleeve (EUR m)'],
              colors=['blue', 'red'], alpha=0.4)

ax2.plot(months, lmt_df['backlog_eur'] / 1e6,
         color='red', linewidth=2, marker='o', linestyle='--', label='Backlog (EUR m)')

# shade suspension months
for i, susp in enumerate(lmt_df['suspension_active']):
    if susp:
        ax2.axvspan(i - 0.5, i + 0.5, color='red', alpha=0.15,
                    label='Suspension active' if i == lmt_df['suspension_active'].values.argmax() else '')

ax2.set_ylabel('EUR million')
ax2.set_title('NAV Composition -- Liquid vs Illiquid Sleeve and Backlog')
ax2.legend(loc='upper right')
ax2.spines['bottom'].set_color('#555555')
ax2.grid(axis='y')

plt.tight_layout()

# ── table ─────────────────────────────────────────────────────────────────────
def highlight_lmt(val):
    if val is True:  return 'color: #b71c1c; font-weight: bold'
    if val is False: return 'color: #1b5e20; font-weight: bold'
    return ''
cols = ['gross_redemption_pct', 'liquid_pct_nav', 'gate_active', 'swing_active', 'suspension_active']

display(lmt_df.set_index(['month'])[cols].style.map(highlight_lmt, subset=['gate_active', 'swing_active', 'suspension_active']))

print(f"\nMonths with gate activated:       {lmt_df['gate_active'].sum()}")
print(f"Months with swing pricing active: {lmt_df['swing_active'].sum()}")
print(f"Months with suspension active:    {lmt_df['suspension_active'].sum()}")

## Summary

| Module | Regulatory reference |
|--------|---------------------|
| Liquidity bucketing -- 10% participation rate convention | AIFMD Annex IV, Del. Reg. 231/2013 Art. 46 |
| Redemption modelling -- lognormal draws, three investor types | Del. Reg. 231/2013 Art. 48 |
| Liquidity coverage -- asset vs liability matching by bucket | Del. Reg. 231/2013 Art. 47 |
| Stress testing -- asset volume shocks + stressed redemptions | Del. Reg. 231/2013 Art. 48 |
| Annex IV reporting -- asset and liability profile by bucket | ESMA/2013/232 |
| LMT simulation -- gate, swing pricing, suspension | AIFMD II Dir. 2024/927/EU Annex V; ESMA34-671404336-1364 |

**Key takeaway:** the structural risk in this fund is not the initial liquidity
profile -- 71% of NAV is liquidatable within 30 days and the fund passes
coverage under all stress scenarios at inception. The risk builds dynamically:
as the gate defers redemptions month after month, the liquid sleeve depletes
while the illiquid private credit sleeve stays fixed. NAV converges toward
the illiquid floor. Once the liquid sleeve is exhausted, the fund cannot
meet redemptions without distressed sales of illiquid assets -- the point
at which suspension becomes unavoidable.

---
*Notebook QRE-38 | Sprint 4 | Quant Risk Engine*
*Regulation: Delegated Regulation 231/2013, AIFMD II Directive 2024/927/EU,*
*ESMA/2013/232, ESMA34-671404336-1364 (Guidelines on LMTs, April 2025)*